# Faruq-v3 AF2 + FFAB2 Parent-Preserving — All Seeds (Kaggle)

Eksperimen lanjutan setelah selective-refinement runtime gagal menemukan kandidat. Parent adalah **AF2FS best checkpoint seed-matched yang sudah selesai**. Parent dibekukan; hanya FFAB2 adapters yang trainable.

Urutan:
1. restore hasil AF2FS 3-seed + `selectivity_analysis.json`;
2. audit frozen-parent per seed;
3. train `AF2FFAPR1` seed 42/123/2026, 30 epoch;
4. keputusan frozen tiga-seed vs parent AF2FS;
5. test tetap locked.

Required Kaggle inputs:
- `faruq-v3-experiment-core-v1` terbaru;
- output/state `Faruq_V3_AF2_FFAB2_All_Seeds_All_Stages_Kaggle`;
- output selective refinement yang memuat `selectivity_analysis.json`.

GPU + Internet ON.


In [ ]:
from pathlib import Path
import importlib,json,os,shutil,subprocess,sys,time,torch,zipfile,hashlib

INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir():
    raise RuntimeError('Kaggle-only notebook')

for name in (
    'af2_spectral_kaggle_manifest.json',
    'faruq-development-v3-grouped.tar.bin',
    'D0_seed42_best.pt','D0_seed123_best.pt','D0_seed2026_best.pt',
):
    matches=sorted(INPUT.rglob(name))
    if len(matches)!=1:
        raise FileNotFoundError(f'Harus ada tepat satu {name}; ditemukan {matches}')
    print('CORE INPUT OK:',name,'->',matches[0])

PRIOR_STATE_NAME='af2-ffab2-all-seeds-all-stages-state.zip'
prior_states=sorted(INPUT.rglob(PRIOR_STATE_NAME))
if len(prior_states)!=1:
    raise FileNotFoundError(f'Harus ada tepat satu {PRIOR_STATE_NAME}; ditemukan {prior_states}')
PRIOR=WORK/'af2-ffab2-parent-source'
if PRIOR.exists(): shutil.rmtree(PRIOR)
PRIOR.mkdir(parents=True)
with zipfile.ZipFile(prior_states[0],'r') as z:
    z.extractall(PRIOR)
print('RESTORED AF2FS SOURCE STATE:',prior_states[0])

# Selectivity analysis can be attached as extracted JSON or inside the prior output ZIP.
direct_selectivity=sorted(INPUT.rglob('selectivity_analysis.json'))
selectivity_zips=sorted(INPUT.rglob('af2-ffab2-selective-refinement-output.zip'))
if len(direct_selectivity)==1:
    SELECTIVITY=direct_selectivity[0]
elif len(direct_selectivity)==0 and len(selectivity_zips)==1:
    SELROOT=WORK/'af2-ffab2-selectivity-source'
    if SELROOT.exists(): shutil.rmtree(SELROOT)
    SELROOT.mkdir(parents=True)
    with zipfile.ZipFile(selectivity_zips[0],'r') as z:
        z.extractall(SELROOT)
    found=sorted(SELROOT.rglob('selectivity_analysis.json'))
    if len(found)!=1:
        raise FileNotFoundError(f'selectivity_analysis.json ambigu/hilang dalam ZIP: {found}')
    SELECTIVITY=found[0]
else:
    raise FileNotFoundError(
        f'Butuh tepat satu selectivity_analysis.json atau satu selective output ZIP; '
        f'json={direct_selectivity}, zip={selectivity_zips}'
    )
selectivity=json.loads(SELECTIVITY.read_text(encoding='utf-8'))
if (
    selectivity.get('format')!='coffee_detector.af2_ffa.selectivity_analysis.v1'
    or selectivity.get('decision')!='NO_RUNTIME_CANDIDATE_PASSES_GATE'
    or selectivity.get('training_authorized') is not False
    or selectivity.get('test_opened') is not False
):
    raise RuntimeError('Selectivity diagnosis tidak cocok dengan follow-up parent-preserving')
print('SELECTIVITY SOURCE OK:',SELECTIVITY)


In [ ]:
BRANCH='codex/af2-ffab2-parent-preserving'
REPO=WORK/'coffee-bean-detection'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(1,4):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,
                      'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)

subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'):
        sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)

COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('BRANCH:',BRANCH)
print('COMMIT:',COMMIT)
print('ULTRALYTICS:',__import__('ultralytics').__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan Kaggle GPU sebelum Run All.')

subprocess.run([
    sys.executable,'-m','pytest','-q',
    'tests/test_af2_ffa.py',
    'tests/test_af2_ffab2_parent_preserving.py',
],cwd=REPO,check=True)
print('PARENT-PRESERVING TESTS PASS')


In [ ]:
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
from coffee_detector.af2_spectral.audit import sha256

DATA,ARTIFACTS,CORE=prepare_af2_spectral_kaggle_input(INPUT,WORK)
if CORE.get('decision')!='PASS' or CORE.get('test_images_accessed') is not False:
    raise RuntimeError('Core contract gagal')
if (DATA/'test').exists():
    raise RuntimeError('TEST TEREXPOSE — STOP')
GROUPED=DATA/'faruq_grouped_summary.json'
SEEDS=(42,123,2026)

def find_one(root, filename):
    matches=sorted(root.rglob(filename))
    if len(matches)!=1:
        raise FileNotFoundError(f'Harus tepat satu {filename}; ditemukan {matches}')
    return matches[0].resolve()

PARENT_RESULTS={s:find_one(PRIOR,f'AF2FS_seed{s}_result.json') for s in SEEDS}

# Resolve best.pt by the SHA stored in each AF2FS result, not stale absolute paths.
best_files=sorted(PRIOR.rglob('best.pt'))
def resolve_parent(seed):
    payload=json.loads(PARENT_RESULTS[seed].read_text(encoding='utf-8'))
    expected=payload['checkpoint_sha256']
    matches=[p.resolve() for p in best_files if sha256(p)==expected]
    if len(matches)!=1:
        raise FileNotFoundError(f'Parent AF2FS seed {seed} SHA {expected} -> {matches}')
    return matches[0]
PARENTS={s:resolve_parent(s) for s in SEEDS}
for s in SEEDS:
    print('AF2FS PARENT',s,':',PARENTS[s],sha256(PARENTS[s]))

OUT=WORK/'af2-ffab2-parent-preserving-v1'
STATE_ZIP=WORK/'af2-ffab2-parent-preserving-state.zip'
prior_parent_states=sorted(INPUT.rglob(STATE_ZIP.name))
if len(prior_parent_states)>1:
    raise RuntimeError(f'Parent-preserving resume state ambigu: {prior_parent_states}')
if len(prior_parent_states)==1 and not OUT.exists():
    print('RESTORE PARENT-PRESERVING STATE:',prior_parent_states[0])
    with zipfile.ZipFile(prior_parent_states[0],'r') as z:
        z.extractall(WORK)
OUT.mkdir(parents=True,exist_ok=True)
print('OUTPUT:',OUT)


In [ ]:
from coffee_detector.af2_ffa import run_af2_ffa_parent_preserving_audit

STATIC={}
for s in SEEDS:
    path=OUT/'static_audits'/f'parent_preserving_static_audit_seed{s}.json'
    path.parent.mkdir(parents=True,exist_ok=True)
    audit=run_af2_ffa_parent_preserving_audit(PARENTS[s],path,device='cuda:0')
    if (
        audit.get('decision')!='PASS'
        or audit.get('training_authorized') is not True
        or audit.get('test_access_authorized') is not False
    ):
        raise RuntimeError(f'STOP: parent-preserving static audit seed {s} gagal: {audit}')
    STATIC[s]=path
    print('STATIC PASS:',s,
          'trainable=',audit['records']['AF2FFAPR1']['trainable_parameters'],
          'of',audit['records']['AF2FFAPR1']['total_parameters'])


In [ ]:
def snapshot_state():
    if STATE_ZIP.exists(): STATE_ZIP.unlink()
    archive=Path(shutil.make_archive(
        str(STATE_ZIP.with_suffix('')),'zip',root_dir=WORK,base_dir=OUT.name
    ))
    print('STATE SNAPSHOT:',archive,archive.stat().st_size,'bytes',flush=True)
    return archive

def validate_result(path,seed):
    payload=json.loads(Path(path).read_text(encoding='utf-8'))
    if payload.get('format')!='coffee_detector.af2_ffa.parent_preserving_arm_result.v1':
        raise RuntimeError(f'Format result salah: {path}')
    if payload.get('arm')!='AF2FFAPR1' or int(payload.get('seed'))!=seed:
        raise RuntimeError(f'Arm/seed salah: {path}')
    if payload.get('parent_frozen') is not True or payload.get('trainable_scope')!='ffab_adapters_only':
        raise RuntimeError(f'Frozen-parent contract hilang: {path}')
    if payload.get('evaluation_split')!='val' or payload.get('test_images_accessed') is not False:
        raise RuntimeError(f'Test-lock/split salah: {path}')
    if payload.get('parent_checkpoint_sha256')!=sha256(PARENTS[seed]):
        raise RuntimeError(f'Parent SHA salah: {path}')
    return payload

def run_candidate(seed):
    result=OUT/'val_reports'/f'AF2FFAPR1_seed{seed}_result.json'
    log=OUT/'logs'/f'AF2FFAPR1_seed{seed}.log'
    log.parent.mkdir(parents=True,exist_ok=True)
    if result.is_file():
        payload=validate_result(result,seed)
        print('REUSE COMPLETE:',seed,{k:payload['metrics'][k] for k in (
            'macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})
        return result

    cmd=[
        sys.executable,'-u','-m',
        'coffee_detector.experiments.run_faruq_v3_af2_ffab2_parent_arm',
        '--arm','AF2FFAPR1',
        '--data-root',str(DATA),
        '--grouped-summary',str(GROUPED),
        '--parent-result',str(PARENT_RESULTS[seed]),
        '--parent-checkpoint',str(PARENTS[seed]),
        '--static-audit',str(STATIC[seed]),
        '--selectivity-analysis',str(SELECTIVITY),
        '--output-root',str(OUT),
        '--seed',str(seed),
        '--device','0',
        '--authorize-training',
    ]
    print('\nSTART/RESUME AF2FFAPR1 seed',seed,'| log=',log,flush=True)
    with log.open('a',encoding='utf-8') as stream:
        p=subprocess.Popen(cmd,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    shown=-1
    while p.poll() is None:
        csv=OUT/'AF2FFAPR1'/f'AF2FFAPR1_seed{seed}'/'results.csv'
        epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epochs!=shown:
            print(f'AF2FFAPR1 seed {seed}: {epochs}/30 epoch tercatat',flush=True)
            shown=epochs
        time.sleep(120)
    if p.returncode:
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-200:]) if log.is_file() else '<no log>'
        raise RuntimeError(f'AF2FFAPR1 seed {seed} gagal, returncode={p.returncode}\n{tail}')
    if not result.is_file():
        raise FileNotFoundError(result)
    payload=validate_result(result,seed)
    print('DONE:',seed,{k:payload['metrics'][k] for k in (
        'macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})
    snapshot_state()
    return result

print('HELPERS READY')


In [ ]:
CANDIDATE_RESULTS=[run_candidate(s) for s in SEEDS]

DECISION=OUT/'val_reports'/'af2_ffab2_parent_preserving_decision.json'
cmd=[
    sys.executable,'-u','-m',
    'coffee_detector.experiments.run_faruq_v3_af2_ffab2_parent_decision',
    '--parent',*[str(PARENT_RESULTS[s]) for s in SEEDS],
    '--candidate',*[str(p) for p in CANDIDATE_RESULTS],
    '--output',str(DECISION),
]
subprocess.run(cmd,cwd=REPO,check=True)
decision=json.loads(DECISION.read_text(encoding='utf-8'))

print('\n================ PARENT-PRESERVING DECISION ================')
print('DECISION:',decision['decision'])
print('NEXT:',decision['next'])
for metric,row in decision['aggregate'].items():
    print(metric,
          'parent=',round(100*row['parent_mean'],3),
          'candidate=',round(100*row['candidate_mean'],3),
          'delta_pp=',round(100*row['delta_mean'],3),
          'improved_seeds=',row['improved_seeds'])
print('CRITERIA:',json.dumps(decision['criteria'],indent=2))
snapshot_state()

FINAL_ZIP=Path(shutil.make_archive(
    str(WORK/'af2-ffab2-parent-preserving-output'),
    'zip',root_dir=WORK,base_dir=OUT.name
))
print('FINAL ZIP:',FINAL_ZIP,FINAL_ZIP.stat().st_size,'bytes')
print('STATE ZIP:',STATE_ZIP)
print('TEST: LOCKED / NEVER OPENED')
